# Exp 03 · Architecture Deep-Dive (1): Action Tokenization Round-Trip

**Goal**: by hand, `encode` a 7-DoF continuous action into tokens, then `decode` it back, and verify the round-trip.

**No training, no GPU.** This whole thing is pure table-lookup + arithmetic (continuous ↔ bin ↔ token_id), unrelated to the 7B model's forward pass — so it runs instantly on CPU.

## Intuition recap (what we already worked out)
1. OpenVLA *is* Llama-2, trained to emit robot actions *as if they were tokens*.
2. 7 continuous numbers → each normalized to `[-1, 1]` → split into **256 bins** → which bin it lands in is an integer 0~255.
3. Those 256 bins map onto Llama's **last 256 rarely-used tokens**: `token_id = vocab_size - bin`.
4. At generation time it's a **for-loop run 7 times**, each step `argmax` emits exactly 1 token, so it's always 7.

This notebook makes points 2 & 3 **tangible** in code.

- **Step 1**: hand-write the binning logic in pure numpy (zero downloads, runs right now).
- **Step 2**: reconcile against OpenVLA's real `ActionTokenizer`, see the actual 256 token IDs (needs `transformers`, still no GPU).

---
## Step 1 · Pure numpy: see the `continuous ↔ bin ↔ token_id` round-trip

The snippet below **faithfully replicates** the core of OpenVLA's `ActionTokenizer`, just without any library.

Conventions (same as official):
- The action is already **normalized to `[-1, 1]`** before it arrives (the normalization stats come from `dataset_statistics.json` — exactly what `unnorm_key` points to in Phase 1!)
- `bins = np.linspace(-1, 1, 256)` cuts `[-1,1]` into 256 uniform boundaries
- `bin_centers` = midpoints between adjacent boundaries (decode uses the bin center as the representative value)
- `token_id = VOCAB_SIZE - bin_index` (reuse the tail of the vocab)

In [1]:
import numpy as np

VOCAB_SIZE = 32000   # Llama-2 vocab size (Step 2 checks this against the real value)
N_BINS     = 256     # 256 bins
ACTION_DIM = 7       # 7-DoF: [dx, dy, dz, droll, dpitch, dyaw, gripper]

bins        = np.linspace(-1.0, 1.0, N_BINS)          # 256 boundaries
bin_centers = (bins[:-1] + bins[1:]) / 2.0            # 255 bin centers

print('first 5 bin boundaries:', bins[:5])
print('bin width ~', bins[1] - bins[0], '(this is the quantization precision: finer differences get washed out)')

first 5 bin boundaries: [-1.         -0.99215686 -0.98431373 -0.97647059 -0.96862745]
bin width ~ 0.007843137254901933 (this is the quantization precision: finer differences get washed out)


In [2]:
def encode(action):
    """continuous action (normalized to [-1,1]) -> 7 token_ids. Pure lookup+arithmetic, no model."""
    action = np.clip(action, -1.0, 1.0)          # clamp out-of-range back into [-1,1]
    bin_idx  = np.digitize(action, bins)          # which bin it falls into (1..255)
    token_id = VOCAB_SIZE - bin_idx               # reuse the tail of the vocab
    return token_id, bin_idx

def decode(token_id):
    """7 token_ids -> continuous action (take bin center as the representative value)."""
    bin_idx = VOCAB_SIZE - token_id
    bin_idx = np.clip(bin_idx - 1, 0, bin_centers.shape[0] - 1)  # align to the centers array
    return bin_centers[bin_idx]

In [3]:
# A 'fake action' to test the round-trip. gripper is deliberately set to 1.0 (fully closed) to see what happens.
action = np.array([0.0237, -0.1500,  0.3812, -0.0050,  0.0900, -0.4001,  1.0])

token_ids, bin_idx = encode(action)
recovered          = decode(token_ids)

labels = ['dx', 'dy', 'dz', 'droll', 'dpitch', 'dyaw', 'gripper']
print(f"{'dim':<8}{'original':>10}{'bin':>8}{'token_id':>12}{'decoded':>12}{'quant_err':>12}")
for l, a, b, t, r in zip(labels, action, bin_idx, token_ids, recovered):
    print(f'{l:<8}{a:>10.4f}{b:>8}{t:>12}{r:>12.4f}{abs(a-r):>12.4f}')

dim       original     bin    token_id     decoded   quant_err
dx          0.0237     131       31869      0.0235      0.0002
dy         -0.1500     109       31891     -0.1490      0.0010
dz          0.3812     177       31823      0.3843      0.0031
droll      -0.0050     127       31873     -0.0078      0.0028
dpitch      0.0900     139       31861      0.0863      0.0037
dyaw       -0.4001      77       31923     -0.4000      0.0001
gripper     1.0000     256       31744      0.9961      0.0039


### Looking at this table, you should confirm 3 things
1. **token_id is always between `31744 ~ 31999`** (= `32000 - 256` to `32000 - 1`) — that's the 'last 256 tokens of the vocab', seen with your own eyes.
2. **'decoded' ≠ 'original', but close** — the gap is the 'quantization error', always less than half a bin width. This is the inherent cost of discretization: continuous values get rounded to the nearest bin center.
3. **gripper=1.0 lands in the last bin** — answering that earlier question: even though the gripper only has open/closed states, the mechanism still treats it as continuous and stuffs it into 256 bins; in practice it only ever occupies the two end bins. **Generic mechanism, no special-casing** — slightly wasteful but harmless.

👉 Change the `action` values above and re-run: what happens if you set one to `2.0` (out of range)? Set one to `0.001` (tiny) — does the quantization error become a bigger fraction?

---
## Step 2 · Reconcile against OpenVLA's real `ActionTokenizer`

Step 1 was our own version. Now run the official logic to verify they agree, and see what the **real token strings** look like.

We only need the Llama tokenizer (a few MB, **no 7B weights, no GPU**). Install the lib first:

```bash
pip install transformers
```

The class below is a trimmed version of OpenVLA repo's `prismatic/vla/action_tokenizer.py` (logic identical to official).

In [4]:
from transformers import AutoTokenizer

# Llama-2 tokenizer (OpenVLA's base). First run downloads a few MB of tokenizer files.
# Using the no-login mirror NousResearch/Llama-2-7b-hf (vocab identical to official, but no HF auth needed).
tok = AutoTokenizer.from_pretrained('NousResearch/Llama-2-7b-hf')
print('real vocab_size =', tok.vocab_size, '(vs the 32000 we assumed in Step 1)')

/Users/duanxingjuan/GitPJ/openVLA-study/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


real vocab_size = 32000 (vs the 32000 we assumed in Step 1)


In [5]:
class ActionTokenizer:
    """Trimmed replica of OpenVLA's official action tokenizer."""
    def __init__(self, tokenizer, bins=256, min_action=-1, max_action=1):
        self.tokenizer = tokenizer
        self.n_bins = bins
        self.bins = np.linspace(min_action, max_action, bins)
        self.bin_centers = (self.bins[:-1] + self.bins[1:]) / 2.0

    def encode(self, action):
        action = np.clip(action, -1, 1)
        disc = np.digitize(action, self.bins)
        return self.tokenizer.vocab_size - disc   # token_id

    def decode(self, token_ids):
        disc = self.tokenizer.vocab_size - token_ids
        disc = np.clip(disc - 1, 0, self.bin_centers.shape[0] - 1)
        return self.bin_centers[disc]

atk = ActionTokenizer(tok)

ids_official = atk.encode(action)
print('official token_id :', ids_official)
print('our Step 1        :', token_ids)
print('match?            ', bool(np.all(ids_official == token_ids)))
print()
print('the 7 tokens decoded into their real strings (rare symbols from the vocab):')
print('   ', [tok.decode([int(i)]) for i in ids_official])

official token_id : [31869 31891 31823 31873 31861 31923 31744]
our Step 1        : [31869 31891 31823 31873 31861 31923 31744]
match?             True

the 7 tokens decoded into their real strings (rare symbols from the vocab):
    ['红', '̍', '喜', 'Έ', 'ರ', 'ᵉ', '忠']


### What you should see in Step 2
- The official token_ids are **identical** to our hand-written ones — proving the logic you understood in Step 1 *is* the official logic.
- Decoding those tokens into strings gives a bunch of **rarely-used symbols** — which is exactly why reusing the vocab tail doesn't hurt the model's ability to talk: these tokens are essentially unused anyway.

---
## ✅ Section summary (backfilled into LEARNING_PLAN.md Phase 2)
- [x] Print the action tokenization: 7-DoF → 256 bins → last 256 vocab tokens
- [x] Hand-roll encode→decode round-trip, verify match + observe quantization error
- [x] Reason through how a binary dim like gripper behaves under the 256-bin mechanism

**Next stop**: the vision side — how does a camera image become something Llama can 'read'? Why two encoders, SigLIP + DINOv2?